# CRM Sales Reports: Tables → Charts → PDF Plan

## What this shows

Build sales-report inputs from deterministic CRM fixtures: pipeline rollups, weekly activity trends, account tables, chart objects, and a PDF report plan. This governed notebook is pure/offline: it does not read CRM credentials, construct Salesforce connectors, create output directories, or render files.


## 1. Fixture CRM tables

The notebook starts from already-extracted fixture tables matching the canonical adapter columns.


In [ ]:
from datetime import datetime

import pandas as pd

opportunities = pd.DataFrame([
    {"Name": "Austin field program", "StageName": "Proposal", "Amount": 125000, "CloseDate": "2026-03-15", "Probability": 45, "CreatedDate": "2026-01-03"},
    {"Name": "Houston GOTV analytics", "StageName": "Negotiation", "Amount": 220000, "CloseDate": "2026-04-01", "Probability": 70, "CreatedDate": "2026-01-09"},
    {"Name": "Volunteer data mart", "StageName": "Closed Won", "Amount": 95000, "CloseDate": "2026-02-20", "Probability": 100, "CreatedDate": "2026-01-16"},
])
accounts = pd.DataFrame([
    {"Name": "Analytical Engines LLC", "Industry": "Civic Tech", "AnnualRevenue": 2500000, "NumberOfEmployees": 42, "BillingCity": "Austin", "BillingState": "TX"},
    {"Name": "Orbital Math Lab", "Industry": "Analytics", "AnnualRevenue": 1750000, "NumberOfEmployees": 25, "BillingCity": "Houston", "BillingState": "TX"},
])
activities = pd.DataFrame([
    {"Subject": "Discovery call", "CreatedDate": "2026-01-03", "ActivityType": "Call"},
    {"Subject": "Proposal sent", "CreatedDate": "2026-01-10", "ActivityType": "Email"},
    {"Subject": "Negotiation sync", "CreatedDate": "2026-01-17", "ActivityType": "Meeting"},
])
print(f"Fixture opportunities: {len(opportunities)}")
print(f"Fixture accounts: {len(accounts)}")
print(f"Fixture activities: {len(activities)}")


## 2. Pipeline visualization shape

`pipeline_adapter` turns opportunity rows into stage-level counts and value totals.


In [ ]:
from siege_utilities.connectors import pipeline_adapter

pipeline = pipeline_adapter(
    opportunities,
    stage_column="StageName",
    value_column="Amount",
    stage_order=["Prospecting", "Proposal", "Negotiation", "Closed Won"],
)
print("Pipeline total value:", int(pipeline["total_value"].sum()))
display(pipeline)


## 3. Chart configs, not files

The standalone chart helpers return renderer-ready chart objects. This notebook validates object creation and report planning only; it does not write images or PDFs.


In [ ]:
from siege_utilities.reporting import create_bar_chart, create_line_chart
from siege_utilities.connectors import timeseries_adapter

pipeline_chart = create_bar_chart(
    data=pipeline,
    x_column="total_value",
    y_column="stage",
    title="Sales Pipeline by Stage",
)
activity_series = timeseries_adapter(activities, timestamp_column="CreatedDate", freq="W", agg="count")
activity_chart = create_line_chart(
    data=activity_series,
    x_column="date",
    y_column="value",
    title="Weekly CRM Activity",
)
print("Pipeline chart object:", type(pipeline_chart).__name__)
print("Activity periods:", len(activity_series))
print("Activity chart object:", type(activity_chart).__name__)


## 4. Top account and opportunity tables

`tabular_adapter` selects, renames, sorts, and limits report-ready tables.


In [ ]:
from siege_utilities.connectors import tabular_adapter

top_accounts = tabular_adapter(
    accounts,
    columns=["Name", "Industry", "AnnualRevenue", "NumberOfEmployees", "BillingCity", "BillingState"],
    rename={"AnnualRevenue": "Annual Revenue", "NumberOfEmployees": "Employees", "BillingCity": "City", "BillingState": "State"},
    sort_by="Annual Revenue",
    ascending=False,
)
top_opportunities = tabular_adapter(
    opportunities,
    columns=["Name", "StageName", "Amount", "CloseDate", "Probability"],
    rename={"StageName": "Stage", "CloseDate": "Close Date"},
    sort_by="Amount",
    ascending=False,
)
print("Top account:", top_accounts.loc[0, "Name"])
print("Top opportunity:", top_opportunities.loc[0, "Name"])
display(top_accounts)
display(top_opportunities)


## 5. PDF report plan

A production script can pass these chart/table objects to report rendering. Governed notebook execution stops at a declarative plan so it remains side-effect-free.


In [ ]:
report_plan = {
    "title": "Sales Pipeline Report",
    "subtitle": "Generated from deterministic CRM fixtures",
    "sections": [
        {"type": "chart", "title": "Pipeline", "payload_type": type(pipeline_chart).__name__},
        {"type": "chart", "title": "Activity", "payload_type": type(activity_chart).__name__},
        {"type": "table", "title": "Top Accounts", "rows": len(top_accounts)},
        {"type": "table", "title": "Top Opportunities", "rows": len(top_opportunities)},
    ],
    "output_policy": "plan only; no files or CRM writes in governed notebook",
}
print(report_plan["title"])
print("Report sections:", [section["title"] for section in report_plan["sections"]])
print(report_plan["output_policy"])


## Related

- Source: `siege_utilities/connectors/_adapters.py`, `siege_utilities/reporting/chart_generator.py`, `siege_utilities/reporting/report_generator.py`
- Tests: `tests/test_connectors_errors.py`, `tests/test_reporting_errors.py`, `tests/test_reporting_module.py`
- Notebook governance: `tests/test_notebook_hygiene.py`, `tests/test_notebooks.py`, `scripts/check_notebook_inventory.py`
